# Silver Layer

## Purpose
Read the Bronze Delta table, clean and validate the data, and write to Silver Delta table.

## Source
- Input Layer: Bronze
- Format: Delta

## Input
/Volumes/workspace/default/nyc_raw_data/bronze

## Output
/Volumes/workspace/default/nyc_raw_data/silver

## Transformations
- Read the Bronze Delta table
- Remove duplicate records
- Handle missing or invalid values
- Filter out incorrect trip records
- Standardize data types where necessary
- Prepare a clean dataset for business analytics.

## Expected Results
The dataset is cleaned, validated, and stored as a Delta table in the Silver layer, ready for downstream transformations and reporting.

---

**Author:** Hudhayfah Shuaib
**Date:** 2026-07-01
**Version:** 1.0
**Description:** This notebook reads the Bronze Delta table, cleans and validates the data, and writes it to the Silver Delta table.


In [0]:
# Read Bronze delta data
bronze_df = spark.read.format("delta").load("/Volumes/workspace/default/nyc_raw_data/bronze/")



In [0]:
# Inspect data

bronze_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [0]:
# Check the Schema

bronze_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [0]:
# Count the rows
bronze_df.count()

3724889

In [0]:
# Check for null values

from pyspark.sql.functions import col, sum

bronze_df.select(
    *[sum(col(c).isNull().cast("int")).alias(c) 
      for c in bronze_df.columns
      ]
).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

In [0]:
# Remove duplicates

silver_df = bronze_df.dropDuplicates()


In [0]:
# Compare counts of bronze_df and silver_df
print(bronze_df.count())
print(silver_df.count())

3724889
3724889


In [0]:
# enforcing business rules

from pyspark.sql.functions import col

silver_df = silver_df.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("total_amount") > 0)
)

In [0]:
# Check row count again

print(bronze_df.count())
print(silver_df.count())

3724889
2551851


In [0]:
# Validate timestamps

silver_df = silver_df.filter(
    (col("tpep_pickup_datetime") < 
     col("tpep_dropoff_datetime"))
)

# Check row count again

print(bronze_df.count())
print(silver_df.count())

3724889
2507959


In [0]:
# Check cleaned data

silver_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2026-01-01 00:55:56|  2026-01-01 01:46:40|              1|          6.8|         1|                 N|         234|    

In [0]:
# confirm schema 

silver_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [0]:
# silver delta table

silver_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/nyc_raw_data/silver")